In [ ]:
#@title Imports

import base64
import io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import scipy.stats as stats
from statsmodels.stats.weightstats import ztest

# Seaborn style - remove outer plot ticks, white plot background.
sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1, rc={
    "lines.linewidth": 1.2,
    "xtick.major.size": 0,
    "xtick.minor.size": 0,
    "ytick.major.size": 0,
    "ytick.minor.size": 0
})

pal = sns.color_palette("Set2")
sns.set_palette(pal)

# Higher resolution plots
%config InlineBackend.figure_format = 'retina'

rater_cols = ['A', 'B', 'C', 'D', 'E']

In [ ]:
#@title Load raw data of Task 1.

realism_df_str = 'UEFSMRUEFSAVJEwVBBUAEgAAEDwAAAAAAAAAAAAAAAAAAPA/FQAVHhUiLBVkFRAVBhUGHBgIAAAAAAAA8D8YCAAAAAAAAACAFgAoCAAAAAAAAPA/GAgAAAAAAAAAgAAAAA84AgAAAGQBAQ8O3X5h534DJuQBHBUKGTUABhAZGAFBFQIWZBbUARbcASZIJggcGAgAAAAAAADwPxgIAAAAAAAAAIAWACgIAAAAAAAA8D8YCAAAAAAAAACAABksFQQVABUCABUAFRAVAgAAABUEFSAVJEwVBBUAEgAAEDwAAAAAAADwPwAAAAAAAAAAFQAVHhUiLBVkFRAVBhUGHBgIAAAAAAAA8D8YCAAAAAAAAACAFgAoCAAAAAAAAPA/GAgAAAAAAAAAgAAAAA84AgAAAGQBAQ/sAQRlDY0AJvYEHBUKGTUABhAZGAFCFQIWZBbUARbcASbaAyaaAxwYCAAAAAAAAPA/GAgAAAAAAAAAgBYAKAgAAAAAAADwPxgIAAAAAAAAAIAAGSwVBBUAFQIAFQAVEBUCAAAAFQQVIBUkTBUEFQASAAAQPAAAAAAAAAAAAAAAAAAA8D8VABUaFR4sFWQVEBUGFQYcGAgAAAAAAADwPxgIAAAAAAAAAIAWACgIAAAAAAAA8D8YCAAAAAAAAACAAAAADTACAAAAZAEBA95KAQMeJogIHBUKGTUABhAZGAFDFQIWZBbQARbYASbwBiawBhwYCAAAAAAAAPA/GAgAAAAAAAAAgBYAKAgAAAAAAADwPxgIAAAAAAAAAIAAGSwVBBUAFQIAFQAVEBUCAAAAFQQVIBUkTBUEFQASAAAQPAAAAAAAAPA/AAAAAAAAAAAVABUeFSIsFWQVEBUGFQYcGAgAAAAAAADwPxgIAAAAAAAAAIAWACgIAAAAAAAA8D8YCAAAAAAAAACAAAAADzgCAAAAZAEBD+wJUUKM6wImngscFQoZNQAGEBkYAUQVAhZkFtQBFtwBJoIKJsIJHBgIAAAAAAAA8D8YCAAAAAAAAACAFgAoCAAAAAAAAPA/GAgAAAAAAAAAgAAZLBUEFQAVAgAVABUQFQIAAAAVBBUgFSRMFQQVABIAABA8AAAAAAAA8D8AAAAAAAAAABUAFR4VIiwVZBUQFQYVBhwYCAAAAAAAAPA/GAgAAAAAAAAAgBYAKAgAAAAAAADwPxgIAAAAAAAAAIAAAAAPOAIAAABkAQEP1m4kmBxxASa0DhwVChk1AAYQGRgBRRUCFmQW1AEW3AEmmA0m2AwcGAgAAAAAAADwPxgIAAAAAAAAAIAWACgIAAAAAAAA8D8YCAAAAAAAAACAABksFQQVABUCABUAFRAVAgAAABUEFVAVPkwVChUAEgAAKAAzBQEM4z+amQEBAOkNCADZDQgkyT8AAAAAAADwPxUAFToVPiwVZBUQFQYVBhwYCAAAAAAAAPA/GAiamZmZmZnJPxYAKAgAAAAAAADwPxgImpmZmZmZyT8AAAAdcAIAAABkAQMPiIBtQECEIJgkAYAAoQYySxBBCQAAJoASHBUKGTUABhAZGARtZWFuFQIWZBagAhaSAibIECbuDxwYCAAAAAAAAPA/GAiamZmZmZnJPxYAKAgAAAAAAADwPxgImpmZmZmZyT8AGSwVBBUAFQIAFQAVEBUCAAAAFQQVKhUuTBUEFQASAAAVUAQAAABSZWFsCQAAAEdlbmVyYXRlZBUAFR4VIiwVZBUQFQYVBhw2ACgEUmVhbBgJR2VuZXJhdGVkAAAADzgCAAAAZAEBD0jvBEO+mAIm+BQcFQwZNQAGEBkYBWxhYmVsFQIWZBawARa4ASaKFCbAExw2ACgEUmVhbBgJR2VuZXJhdGVkABksFQQVABUCABUAFRAVAgAAABUEGYw1ABgGc2NoZW1hFQ4AFQolAhgBQQAVCiUCGAFCABUKJQIYAUMAFQolAhgBRAAVCiUCGAFFABUKJQIYBG1lYW4AFQwlAhgFbGFiZWwlAEwcAAAAFmQZHBl8JuQBHBUKGTUABhAZGAFBFQIWZBbUARbcASZIJggcGAgAAAAAAADwPxgIAAAAAAAAAIAWACgIAAAAAAAA8D8YCAAAAAAAAACAABksFQQVABUCABUAFRAVAgAAACb2BBwVChk1AAYQGRgBQhUCFmQW1AEW3AEm2gMmmgMcGAgAAAAAAADwPxgIAAAAAAAAAIAWACgIAAAAAAAA8D8YCAAAAAAAAACAABksFQQVABUCABUAFRAVAgAAACaICBwVChk1AAYQGRgBQxUCFmQW0AEW2AEm8AYmsAYcGAgAAAAAAADwPxgIAAAAAAAAAIAWACgIAAAAAAAA8D8YCAAAAAAAAACAABksFQQVABUCABUAFRAVAgAAACaeCxwVChk1AAYQGRgBRBUCFmQW1AEW3AEmggomwgkcGAgAAAAAAADwPxgIAAAAAAAAAIAWACgIAAAAAAAA8D8YCAAAAAAAAACAABksFQQVABUCABUAFRAVAgAAACa0DhwVChk1AAYQGRgBRRUCFmQW1AEW3AEmmA0m2AwcGAgAAAAAAADwPxgIAAAAAAAAAIAWACgIAAAAAAAA8D8YCAAAAAAAAACAABksFQQVABUCABUAFRAVAgAAACaAEhwVChk1AAYQGRgEbWVhbhUCFmQWoAIWkgImyBAm7g8cGAgAAAAAAADwPxgImpmZmZmZyT8WACgIAAAAAAAA8D8YCJqZmZmZmck/ABksFQQVABUCABUAFRAVAgAAACb4FBwVDBk1AAYQGRgFbGFiZWwVAhZkFrABFrgBJooUJsATHDYAKARSZWFsGAlHZW5lcmF0ZWQAGSwVBBUAFQIAFQAVEBUCAAAAFvALFmQmCBaSDBQAABksGAZwYW5kYXMYogh7ImluZGV4X2NvbHVtbnMiOiBbeyJraW5kIjogInJhbmdlIiwgIm5hbWUiOiBudWxsLCAic3RhcnQiOiAwLCAic3RvcCI6IDUwLCAic3RlcCI6IDF9XSwgImNvbHVtbl9pbmRleGVzIjogW3sibmFtZSI6IG51bGwsICJmaWVsZF9uYW1lIjogbnVsbCwgInBhbmRhc190eXBlIjogInVuaWNvZGUiLCAibnVtcHlfdHlwZSI6ICJvYmplY3QiLCAibWV0YWRhdGEiOiB7ImVuY29kaW5nIjogIlVURi04In19XSwgImNvbHVtbnMiOiBbeyJuYW1lIjogIkEiLCAiZmllbGRfbmFtZSI6ICJBIiwgInBhbmRhc190eXBlIjogImZsb2F0NjQiLCAibnVtcHlfdHlwZSI6ICJmbG9hdDY0IiwgIm1ldGFkYXRhIjogbnVsbH0sIHsibmFtZSI6ICJCIiwgImZpZWxkX25hbWUiOiAiQiIsICJwYW5kYXNfdHlwZSI6ICJmbG9hdDY0IiwgIm51bXB5X3R5cGUiOiAiZmxvYXQ2NCIsICJtZXRhZGF0YSI6IG51bGx9LCB7Im5hbWUiOiAiQyIsICJmaWVsZF9uYW1lIjogIkMiLCAicGFuZGFzX3R5cGUiOiAiZmxvYXQ2NCIsICJudW1weV90eXBlIjogImZsb2F0NjQiLCAibWV0YWRhdGEiOiBudWxsfSwgeyJuYW1lIjogIkQiLCAiZmllbGRfbmFtZSI6ICJEIiwgInBhbmRhc190eXBlIjogImZsb2F0NjQiLCAibnVtcHlfdHlwZSI6ICJmbG9hdDY0IiwgIm1ldGFkYXRhIjogbnVsbH0sIHsibmFtZSI6ICJFIiwgImZpZWxkX25hbWUiOiAiRSIsICJwYW5kYXNfdHlwZSI6ICJmbG9hdDY0IiwgIm51bXB5X3R5cGUiOiAiZmxvYXQ2NCIsICJtZXRhZGF0YSI6IG51bGx9LCB7Im5hbWUiOiAibWVhbiIsICJmaWVsZF9uYW1lIjogIm1lYW4iLCAicGFuZGFzX3R5cGUiOiAiZmxvYXQ2NCIsICJudW1weV90eXBlIjogImZsb2F0NjQiLCAibWV0YWRhdGEiOiBudWxsfSwgeyJuYW1lIjogImxhYmVsIiwgImZpZWxkX25hbWUiOiAibGFiZWwiLCAicGFuZGFzX3R5cGUiOiAidW5pY29kZSIsICJudW1weV90eXBlIjogIm9iamVjdCIsICJtZXRhZGF0YSI6IG51bGx9XSwgImNyZWF0b3IiOiB7ImxpYnJhcnkiOiAicHlhcnJvdyIsICJ2ZXJzaW9uIjogIjEzLjAuMCJ9LCAicGFuZGFzX3ZlcnNpb24iOiAiMi4wLjMifQAYDEFSUk9XOnNjaGVtYRjYDy8vLy8vOWdGQUFBUUFBQUFBQUFLQUE0QUJnQUZBQWdBQ2dBQUFBQUJCQUFRQUFBQUFBQUtBQXdBQUFBRUFBZ0FDZ0FBQUZnRUFBQUVBQUFBQVFBQUFBd0FBQUFJQUF3QUJBQUlBQWdBQUFBd0JBQUFCQUFBQUNJRUFBQjdJbWx1WkdWNFgyTnZiSFZ0Ym5NaU9pQmJleUpyYVc1a0lqb2dJbkpoYm1kbElpd2dJbTVoYldVaU9pQnVkV3hzTENBaWMzUmhjblFpT2lBd0xDQWljM1J2Y0NJNklEVXdMQ0FpYzNSbGNDSTZJREY5WFN3Z0ltTnZiSFZ0Ymw5cGJtUmxlR1Z6SWpvZ1czc2libUZ0WlNJNklHNTFiR3dzSUNKbWFXVnNaRjl1WVcxbElqb2diblZzYkN3Z0luQmhibVJoYzE5MGVYQmxJam9nSW5WdWFXTnZaR1VpTENBaWJuVnRjSGxmZEhsd1pTSTZJQ0p2WW1wbFkzUWlMQ0FpYldWMFlXUmhkR0VpT2lCN0ltVnVZMjlrYVc1bklqb2dJbFZVUmkwNEluMTlYU3dnSW1OdmJIVnRibk1pT2lCYmV5SnVZVzFsSWpvZ0lrRWlMQ0FpWm1sbGJHUmZibUZ0WlNJNklDSkJJaXdnSW5CaGJtUmhjMTkwZVhCbElqb2dJbVpzYjJGME5qUWlMQ0FpYm5WdGNIbGZkSGx3WlNJNklDSm1iRzloZERZMElpd2dJbTFsZEdGa1lYUmhJam9nYm5Wc2JIMHNJSHNpYm1GdFpTSTZJQ0pDSWl3Z0ltWnBaV3hrWDI1aGJXVWlPaUFpUWlJc0lDSndZVzVrWVhOZmRIbHdaU0k2SUNKbWJHOWhkRFkwSWl3Z0ltNTFiWEI1WDNSNWNHVWlPaUFpWm14dllYUTJOQ0lzSUNKdFpYUmhaR0YwWVNJNklHNTFiR3g5TENCN0ltNWhiV1VpT2lBaVF5SXNJQ0ptYVdWc1pGOXVZVzFsSWpvZ0lrTWlMQ0FpY0dGdVpHRnpYM1I1Y0dVaU9pQWlabXh2WVhRMk5DSXNJQ0p1ZFcxd2VWOTBlWEJsSWpvZ0ltWnNiMkYwTmpRaUxDQWliV1YwWVdSaGRHRWlPaUJ1ZFd4c2ZTd2dleUp1WVcxbElqb2dJa1FpTENBaVptbGxiR1JmYm1GdFpTSTZJQ0pFSWl3Z0luQmhibVJoYzE5MGVYQmxJam9nSW1ac2IyRjBOalFpTENBaWJuVnRjSGxmZEhsd1pTSTZJQ0ptYkc5aGREWTBJaXdnSW0xbGRHRmtZWFJoSWpvZ2JuVnNiSDBzSUhzaWJtRnRaU0k2SUNKRklpd2dJbVpwWld4a1gyNWhiV1VpT2lBaVJTSXNJQ0p3WVc1a1lYTmZkSGx3WlNJNklDSm1iRzloZERZMElpd2dJbTUxYlhCNVgzUjVjR1VpT2lBaVpteHZZWFEyTkNJc0lDSnRaWFJoWkdGMFlTSTZJRzUxYkd4OUxDQjdJbTVoYldVaU9pQWliV1ZoYmlJc0lDSm1hV1ZzWkY5dVlXMWxJam9nSW0xbFlXNGlMQ0FpY0dGdVpHRnpYM1I1Y0dVaU9pQWlabXh2WVhRMk5DSXNJQ0p1ZFcxd2VWOTBlWEJsSWpvZ0ltWnNiMkYwTmpRaUxDQWliV1YwWVdSaGRHRWlPaUJ1ZFd4c2ZTd2dleUp1WVcxbElqb2dJbXhoWW1Wc0lpd2dJbVpwWld4a1gyNWhiV1VpT2lBaWJHRmlaV3dpTENBaWNHRnVaR0Z6WDNSNWNHVWlPaUFpZFc1cFkyOWtaU0lzSUNKdWRXMXdlVjkwZVhCbElqb2dJbTlpYW1WamRDSXNJQ0p0WlhSaFpHRjBZU0k2SUc1MWJHeDlYU3dnSW1OeVpXRjBiM0lpT2lCN0lteHBZbkpoY25raU9pQWljSGxoY25KdmR5SXNJQ0oyWlhKemFXOXVJam9nSWpFekxqQXVNQ0o5TENBaWNHRnVaR0Z6WDNabGNuTnBiMjRpT2lBaU1pNHdMak1pZlFBQUJnQUFBSEJoYm1SaGN3QUFCd0FBQUNRQkFBRG9BQUFBdkFBQUFKQUFBQUJrQUFBQU5BQUFBQVFBQUFBSS8vLy9BQUFCQlJBQUFBQWNBQUFBQkFBQUFBQUFBQUFGQUFBQWJHRmlaV3dBQUFBRUFBUUFCQUFBQURULy8vOEFBQUVERUFBQUFCZ0FBQUFFQUFBQUFBQUFBQVFBQUFCdFpXRnVBQUFBQUNyLy8vOEFBQUlBWVAvLy93QUFBUU1RQUFBQUZBQUFBQVFBQUFBQUFBQUFBUUFBQUVVQUFBQlMvLy8vQUFBQ0FJai8vLzhBQUFFREVBQUFBQlFBQUFBRUFBQUFBQUFBQUFFQUFBQkVBQUFBZXYvLy93QUFBZ0N3Ly8vL0FBQUJBeEFBQUFBVUFBQUFCQUFBQUFBQUFBQUJBQUFBUXdBQUFLTC8vLzhBQUFJQTJQLy8vd0FBQVFNUUFBQUFGQUFBQUFRQUFBQUFBQUFBQVFBQUFFSUFBQURLLy8vL0FBQUNBQkFBRkFBSUFBWUFCd0FNQUFBQUVBQVFBQUFBQUFBQkF4QUFBQUFZQUFBQUJBQUFBQUFBQUFBQkFBQUFRUUFHQUFnQUJnQUdBQUFBQUFBQ0FBPT0AGCBwYXJxdWV0LWNwcC1hcnJvdyB2ZXJzaW9uIDEzLjAuMBl8HAAAHAAAHAAAHAAAHAAAHAAAHAAAADYPAABQQVIx'
realism_df = pd.read_parquet(io.BytesIO(base64.b64decode(realism_df_str)))
del realism_df_str
realism_df.info()
realism_df.head()

In [ ]:
#@title Compute the accuracy and F1 score of each rater, and the average F1 score across all the raters.

f1_list = []
for rater in rater_cols:
  tp = np.where((realism_df[rater] == 1.) & (realism_df['label'] == 'Real'))[0].size
  tn = np.where((realism_df[rater] == 0.) & (realism_df['label'] == 'Generated'))[0].size
  fp = np.where((realism_df[rater] == 1.) & (realism_df['label'] == 'Generated'))[0].size
  fn = np.where((realism_df[rater] == 0.) & (realism_df['label'] == 'Real'))[0].size
  acc = (tp + tn) / (tp + tn + fp + fn)
  f1 = 2 * tp / (2 * tp + fp + fn)
  print(f'rater: {rater}, accuracy: {acc}, f1: {f1}')

  f1_list.append(f1)

print(f'f1 mean: {np.mean(f1_list)}')
print(f'f1 std: {np.std(f1_list)}')

In [ ]:
#@title Aggregate the raw data to draw figures

realism_df_val_counts = realism_df[['label', 'mean']].value_counts().reset_index()
realism_df_val_counts = realism_df_val_counts.rename({0: 'proportion'}, axis=1)
realism_df_val_counts = realism_df_val_counts[['mean', 'proportion', 'label']]
generated_num = np.where(realism_df['label'] == 'Generated')[0].size
real_num = np.where(realism_df['label'] == 'Real')[0].size

def calc_proportion(row):
  if row['label'] == 'Generated':
    row['proportion'] /= generated_num
  else:
    row['proportion'] /= real_num

  return row

realism_df_val_counts = realism_df_val_counts.apply(lambda x: calc_proportion(x), axis=1)


missing_data = [
    dict(label='Generated', mean=0.0, proportion=0.0),
    dict(label='Real', mean=0.0, proportion=0.0)]

for md in missing_data:
  realism_df_val_counts = pd.concat([realism_df_val_counts, pd.DataFrame([md])], ignore_index=True)

realism_df_val_counts = realism_df_val_counts[['mean', 'proportion', 'label']]

In [ ]:
#@title Figure 4 (A.1)

realism_plot_data = [realism_df[realism_df['label'] == 'Generated']['mean'].to_numpy(), realism_df[realism_df['label'] == 'Real']['mean'].to_numpy()]
zvalue, pvalue = ztest(realism_plot_data[0], realism_plot_data[1], value=0)
print(f'Significance test for the difference between the real and generated samples: z={zvalue}, p={pvalue}')

fig = plt.figure(figsize =(5, 3))

# Creating axes instance
ax = fig.add_subplot(111)

# Creating plot
bp = sns.boxplot(realism_plot_data, medianprops={"alpha": 0.})

ax.set_xticklabels(['Generated', 'Real'], fontdict={'fontsize': 10, 'fontweight': 'heavy'})
ax.set_ylim(-0.05, 1.05)


ax.set_ylabel('Mean sample rating', fontdict={'fontsize': 10, 'fontweight': 'heavy'})
ax.set_title('(A.1)', fontdict={'fontsize': 10, 'fontweight': 'heavy'})
ax.set_xlabel('Sample type', fontdict={'fontsize': 10, 'fontweight': 'heavy'})


In [ ]:
#@title Figure 4 (A.2)

fig = plt.figure(figsize =(5, 3))
ax = fig.add_subplot(111)
bins = [0.1, 0.3, 0.5, 0.7, 0.9, 1.1]
_df = realism_df_val_counts[['mean', 'label', 'proportion']]

hue_order = ['Generated', 'Real']

bp = sns.barplot(_df, x='mean', y='proportion', hue='label', hue_order=hue_order)
ax.get_legend().set_title('Sample type')

ax.xaxis.grid(False)
ax.set_ylabel('Sample proportion', fontdict={'fontsize': 10, 'fontweight': 'heavy'})
ax.set_title('(A.2)', fontdict={'fontsize': 10, 'fontweight': 'heavy'})
ax.set_xlabel('Mean rating value', fontdict={'fontsize': 10, 'fontweight': 'heavy'})


In [ ]:
#@title Load raw data of Task 2

# 1 stands for that the at least one receiver predicted by TacticAI matches the predictions by human raters, 0 otherwise.

receiver_df_str = 'UEFSMRUEFSAVJEwVBBUAEgAAEDwBAAAAAAAAAAAAAAAAAAAAFQAVHBUgLBVkFRAVBhUGHBgIAQAAAAAAAAAYCAAAAAAAAAAAFgAoCAEAAAAAAAAAGAgAAAAAAAAAAAAAAA40AgAAAGQBASYACQMIEAAm4gEcFQQZNQAGEBkYAUEVAhZkFtIBFtoBJkgmCBwYCAEAAAAAAAAAGAgAAAAAAAAAABYAKAgBAAAAAAAAABgIAAAAAAAAAAAAGSwVBBUAFQIAFQAVEBUCAAAAFQQVIBUkTBUEFQASAAAQPAAAAAAAAAAAAQAAAAAAAAAVABUeFSIsFWQVEBUGFQYcGAgBAAAAAAAAABgIAAAAAAAAAAAWACgIAQAAAAAAAAAYCAAAAAAAAAAAAAAADzgCAAAAZAEBD9RGRnflbAAm9AQcFQQZNQAGEBkYAUIVAhZkFtQBFtwBJtgDJpgDHBgIAQAAAAAAAAAYCAAAAAAAAAAAFgAoCAEAAAAAAAAAGAgAAAAAAAAAAAAZLBUEFQAVAgAVABUQFQIAAAAVBBUgFSRMFQQVABIAABA8AQAAAAAAAAAAAAAAAAAAABUAFRYVGiwVZBUQFQYVBhwYCAEAAAAAAAAAGAgAAAAAAAAAABYAKAgBAAAAAAAAABgIAAAAAAAAAAAAAAALKAIAAABkAQFeAAMBJoIIHBUEGTUABhAZGAFDFQIWZBbMARbUASbuBiauBhwYCAEAAAAAAAAAGAgAAAAAAAAAABYAKAgBAAAAAAAAABgIAAAAAAAAAAAAGSwVBBUAFQIAFQAVEBUCAAAAFQQVIBUkTBUEFQASAAAQPAEAAAAAAAAAAAAAAAAAAAAVABUeFSIsFWQVEBUGFQYcGAgBAAAAAAAAABgIAAAAAAAAAAAWACgIAQAAAAAAAAAYCAAAAAAAAAAAAAAADzgCAAAAZAEBD8Q4UAhaCQImmAscFQQZNQAGEBkYAUQVAhZkFtQBFtwBJvwJJrwJHBgIAQAAAAAAAAAYCAAAAAAAAAAAFgAoCAEAAAAAAAAAGAgAAAAAAAAAAAAZLBUEFQAVAgAVABUQFQIAAAAVBBUgFSRMFQQVABIAABA8AQAAAAAAAAAAAAAAAAAAABUAFR4VIiwVZBUQFQYVBhwYCAEAAAAAAAAAGAgAAAAAAAAAABYAKAgBAAAAAAAAABgIAAAAAAAAAAAAAAAPOAIAAABkAQEHBIxAMgACASauDhwVBBk1AAYQGRgBRRUCFmQW1AEW3AEmkg0m0gwcGAgBAAAAAAAAABgIAAAAAAAAAAAWACgIAQAAAAAAAAAYCAAAAAAAAAAAABksFQQVABUCABUAFRAVAgAAABUEFSoVLkwVBBUAEgAAFVAJAAAAR2VuZXJhdGVkBAAAAFJlYWwVABUeFSIsFWQVEBUGFQYcNgAoBFJlYWwYCUdlbmVyYXRlZAAAAA84AgAAAGQBAQ9OemY3XYcAJqARHBUMGTUABhAZGAVsYWJlbBUCFmQWsAEWuAEmshAm6A8cNgAoBFJlYWwYCUdlbmVyYXRlZAAZLBUEFQAVAgAVABUQFQIAAAAVBBVAFT5MFQgVABIAACAEmpkBAQjpPzMFAUTjPwAAAAAAAPA/mpmZmZmZ2T8VABUsFTAsFWQVEBUGFQYcGAgAAAAAAADwPxgImpmZmZmZ2T8WACgIAAAAAAAA8D8YCJqZmZmZmdk/AAAAFlQCAAAAZAECDxACyGVoE2oKZgkhaAwAJrgUHBUKGTUABhAZGARtZWFuFQIWZBaCAhaEAiaOEya0EhwYCAAAAAAAAPA/GAiamZmZmZnZPxYAKAgAAAAAAADwPxgImpmZmZmZ2T8AGSwVBBUAFQIAFQAVEBUCAAAAFQQZjDUAGAZzY2hlbWEVDgAVBCUCGAFBABUEJQIYAUIAFQQlAhgBQwAVBCUCGAFEABUEJQIYAUUAFQwlAhgFbGFiZWwlAEwcAAAAFQolAhgEbWVhbgAWZBkcGXwm4gEcFQQZNQAGEBkYAUEVAhZkFtIBFtoBJkgmCBwYCAEAAAAAAAAAGAgAAAAAAAAAABYAKAgBAAAAAAAAABgIAAAAAAAAAAAAGSwVBBUAFQIAFQAVEBUCAAAAJvQEHBUEGTUABhAZGAFCFQIWZBbUARbcASbYAyaYAxwYCAEAAAAAAAAAGAgAAAAAAAAAABYAKAgBAAAAAAAAABgIAAAAAAAAAAAAGSwVBBUAFQIAFQAVEBUCAAAAJoIIHBUEGTUABhAZGAFDFQIWZBbMARbUASbuBiauBhwYCAEAAAAAAAAAGAgAAAAAAAAAABYAKAgBAAAAAAAAABgIAAAAAAAAAAAAGSwVBBUAFQIAFQAVEBUCAAAAJpgLHBUEGTUABhAZGAFEFQIWZBbUARbcASb8CSa8CRwYCAEAAAAAAAAAGAgAAAAAAAAAABYAKAgBAAAAAAAAABgIAAAAAAAAAAAAGSwVBBUAFQIAFQAVEBUCAAAAJq4OHBUEGTUABhAZGAFFFQIWZBbUARbcASaSDSbSDBwYCAEAAAAAAAAAGAgAAAAAAAAAABYAKAgBAAAAAAAAABgIAAAAAAAAAAAAGSwVBBUAFQIAFQAVEBUCAAAAJqARHBUMGTUABhAZGAVsYWJlbBUCFmQWsAEWuAEmshAm6A8cNgAoBFJlYWwYCUdlbmVyYXRlZAAZLBUEFQAVAgAVABUQFQIAAAAmuBQcFQoZNQAGEBkYBG1lYW4VAhZkFoICFoQCJo4TJrQSHBgIAAAAAAAA8D8YCJqZmZmZmdk/FgAoCAAAAAAAAPA/GAiamZmZmZnZPwAZLBUEFQAVAgAVABUQFQIAAAAWzAsWZCYIFv4LFAAAGSwYBnBhbmRhcxiOCHsiaW5kZXhfY29sdW1ucyI6IFt7ImtpbmQiOiAicmFuZ2UiLCAibmFtZSI6IG51bGwsICJzdGFydCI6IDAsICJzdG9wIjogNTAsICJzdGVwIjogMX1dLCAiY29sdW1uX2luZGV4ZXMiOiBbeyJuYW1lIjogbnVsbCwgImZpZWxkX25hbWUiOiBudWxsLCAicGFuZGFzX3R5cGUiOiAidW5pY29kZSIsICJudW1weV90eXBlIjogIm9iamVjdCIsICJtZXRhZGF0YSI6IHsiZW5jb2RpbmciOiAiVVRGLTgifX1dLCAiY29sdW1ucyI6IFt7Im5hbWUiOiAiQSIsICJmaWVsZF9uYW1lIjogIkEiLCAicGFuZGFzX3R5cGUiOiAiaW50NjQiLCAibnVtcHlfdHlwZSI6ICJpbnQ2NCIsICJtZXRhZGF0YSI6IG51bGx9LCB7Im5hbWUiOiAiQiIsICJmaWVsZF9uYW1lIjogIkIiLCAicGFuZGFzX3R5cGUiOiAiaW50NjQiLCAibnVtcHlfdHlwZSI6ICJpbnQ2NCIsICJtZXRhZGF0YSI6IG51bGx9LCB7Im5hbWUiOiAiQyIsICJmaWVsZF9uYW1lIjogIkMiLCAicGFuZGFzX3R5cGUiOiAiaW50NjQiLCAibnVtcHlfdHlwZSI6ICJpbnQ2NCIsICJtZXRhZGF0YSI6IG51bGx9LCB7Im5hbWUiOiAiRCIsICJmaWVsZF9uYW1lIjogIkQiLCAicGFuZGFzX3R5cGUiOiAiaW50NjQiLCAibnVtcHlfdHlwZSI6ICJpbnQ2NCIsICJtZXRhZGF0YSI6IG51bGx9LCB7Im5hbWUiOiAiRSIsICJmaWVsZF9uYW1lIjogIkUiLCAicGFuZGFzX3R5cGUiOiAiaW50NjQiLCAibnVtcHlfdHlwZSI6ICJpbnQ2NCIsICJtZXRhZGF0YSI6IG51bGx9LCB7Im5hbWUiOiAibGFiZWwiLCAiZmllbGRfbmFtZSI6ICJsYWJlbCIsICJwYW5kYXNfdHlwZSI6ICJ1bmljb2RlIiwgIm51bXB5X3R5cGUiOiAib2JqZWN0IiwgIm1ldGFkYXRhIjogbnVsbH0sIHsibmFtZSI6ICJtZWFuIiwgImZpZWxkX25hbWUiOiAibWVhbiIsICJwYW5kYXNfdHlwZSI6ICJmbG9hdDY0IiwgIm51bXB5X3R5cGUiOiAiZmxvYXQ2NCIsICJtZXRhZGF0YSI6IG51bGx9XSwgImNyZWF0b3IiOiB7ImxpYnJhcnkiOiAicHlhcnJvdyIsICJ2ZXJzaW9uIjogIjEzLjAuMCJ9LCAicGFuZGFzX3ZlcnNpb24iOiAiMi4wLjMifQAYDEFSUk9XOnNjaGVtYRjgDy8vLy8vK0FGQUFBUUFBQUFBQUFLQUE0QUJnQUZBQWdBQ2dBQUFBQUJCQUFRQUFBQUFBQUtBQXdBQUFBRUFBZ0FDZ0FBQUVRRUFBQUVBQUFBQVFBQUFBd0FBQUFJQUF3QUJBQUlBQWdBQUFBY0JBQUFCQUFBQUE0RUFBQjdJbWx1WkdWNFgyTnZiSFZ0Ym5NaU9pQmJleUpyYVc1a0lqb2dJbkpoYm1kbElpd2dJbTVoYldVaU9pQnVkV3hzTENBaWMzUmhjblFpT2lBd0xDQWljM1J2Y0NJNklEVXdMQ0FpYzNSbGNDSTZJREY5WFN3Z0ltTnZiSFZ0Ymw5cGJtUmxlR1Z6SWpvZ1czc2libUZ0WlNJNklHNTFiR3dzSUNKbWFXVnNaRjl1WVcxbElqb2diblZzYkN3Z0luQmhibVJoYzE5MGVYQmxJam9nSW5WdWFXTnZaR1VpTENBaWJuVnRjSGxmZEhsd1pTSTZJQ0p2WW1wbFkzUWlMQ0FpYldWMFlXUmhkR0VpT2lCN0ltVnVZMjlrYVc1bklqb2dJbFZVUmkwNEluMTlYU3dnSW1OdmJIVnRibk1pT2lCYmV5SnVZVzFsSWpvZ0lrRWlMQ0FpWm1sbGJHUmZibUZ0WlNJNklDSkJJaXdnSW5CaGJtUmhjMTkwZVhCbElqb2dJbWx1ZERZMElpd2dJbTUxYlhCNVgzUjVjR1VpT2lBaWFXNTBOalFpTENBaWJXVjBZV1JoZEdFaU9pQnVkV3hzZlN3Z2V5SnVZVzFsSWpvZ0lrSWlMQ0FpWm1sbGJHUmZibUZ0WlNJNklDSkNJaXdnSW5CaGJtUmhjMTkwZVhCbElqb2dJbWx1ZERZMElpd2dJbTUxYlhCNVgzUjVjR1VpT2lBaWFXNTBOalFpTENBaWJXVjBZV1JoZEdFaU9pQnVkV3hzZlN3Z2V5SnVZVzFsSWpvZ0lrTWlMQ0FpWm1sbGJHUmZibUZ0WlNJNklDSkRJaXdnSW5CaGJtUmhjMTkwZVhCbElqb2dJbWx1ZERZMElpd2dJbTUxYlhCNVgzUjVjR1VpT2lBaWFXNTBOalFpTENBaWJXVjBZV1JoZEdFaU9pQnVkV3hzZlN3Z2V5SnVZVzFsSWpvZ0lrUWlMQ0FpWm1sbGJHUmZibUZ0WlNJNklDSkVJaXdnSW5CaGJtUmhjMTkwZVhCbElqb2dJbWx1ZERZMElpd2dJbTUxYlhCNVgzUjVjR1VpT2lBaWFXNTBOalFpTENBaWJXVjBZV1JoZEdFaU9pQnVkV3hzZlN3Z2V5SnVZVzFsSWpvZ0lrVWlMQ0FpWm1sbGJHUmZibUZ0WlNJNklDSkZJaXdnSW5CaGJtUmhjMTkwZVhCbElqb2dJbWx1ZERZMElpd2dJbTUxYlhCNVgzUjVjR1VpT2lBaWFXNTBOalFpTENBaWJXVjBZV1JoZEdFaU9pQnVkV3hzZlN3Z2V5SnVZVzFsSWpvZ0lteGhZbVZzSWl3Z0ltWnBaV3hrWDI1aGJXVWlPaUFpYkdGaVpXd2lMQ0FpY0dGdVpHRnpYM1I1Y0dVaU9pQWlkVzVwWTI5a1pTSXNJQ0p1ZFcxd2VWOTBlWEJsSWpvZ0ltOWlhbVZqZENJc0lDSnRaWFJoWkdGMFlTSTZJRzUxYkd4OUxDQjdJbTVoYldVaU9pQWliV1ZoYmlJc0lDSm1hV1ZzWkY5dVlXMWxJam9nSW0xbFlXNGlMQ0FpY0dGdVpHRnpYM1I1Y0dVaU9pQWlabXh2WVhRMk5DSXNJQ0p1ZFcxd2VWOTBlWEJsSWpvZ0ltWnNiMkYwTmpRaUxDQWliV1YwWVdSaGRHRWlPaUJ1ZFd4c2ZWMHNJQ0pqY21WaGRHOXlJam9nZXlKc2FXSnlZWEo1SWpvZ0luQjVZWEp5YjNjaUxDQWlkbVZ5YzJsdmJpSTZJQ0l4TXk0d0xqQWlmU3dnSW5CaGJtUmhjMTkyWlhKemFXOXVJam9nSWpJdU1DNHpJbjBBQUFZQUFBQndZVzVrWVhNQUFBY0FBQUE0QVFBQStBQUFBTWdBQUFDWUFBQUFhQUFBQURnQUFBQUVBQUFBOVA3Ly93QUFBUU1RQUFBQUhBQUFBQVFBQUFBQUFBQUFCQUFBQUcxbFlXNEFBQVlBQ0FBR0FBWUFBQUFBQUFJQUpQLy8vd0FBQVFVUUFBQUFIQUFBQUFRQUFBQUFBQUFBQlFBQUFHeGhZbVZzQUFBQUJBQUVBQVFBQUFCUS8vLy9BQUFCQWhBQUFBQVVBQUFBQkFBQUFBQUFBQUFCQUFBQVJRQUFBRUQvLy84QUFBQUJRQUFBQUh6Ly8vOEFBQUVDRUFBQUFCUUFBQUFFQUFBQUFBQUFBQUVBQUFCRUFBQUFiUC8vL3dBQUFBRkFBQUFBcVAvLy93QUFBUUlRQUFBQUZBQUFBQVFBQUFBQUFBQUFBUUFBQUVNQUFBQ1kvLy8vQUFBQUFVQUFBQURVLy8vL0FBQUJBaEFBQUFBVUFBQUFCQUFBQUFBQUFBQUJBQUFBUWdBQUFNVC8vLzhBQUFBQlFBQUFBQkFBRkFBSUFBWUFCd0FNQUFBQUVBQVFBQUFBQUFBQkFoQUFBQUFjQUFBQUJBQUFBQUFBQUFBQkFBQUFRUUFBQUFnQURBQUlBQWNBQ0FBQUFBQUFBQUZBQUFBQQAYIHBhcnF1ZXQtY3BwLWFycm93IHZlcnNpb24gMTMuMC4wGXwcAAAcAAAcAAAcAAAcAAAcAAAcAAAAKg8AAFBBUjE='
receiver_df = pd.read_parquet(io.BytesIO(base64.b64decode(receiver_df_str)))
del receiver_df_str
receiver_df.info()
receiver_df.head()

In [ ]:
#@title Prediction accuracies for real and synthetic samples

for label in ['Real', 'Generated']:
  tmp_df = receiver_df[receiver_df['label'] == label]
  rater_accs = []

  for rater in rater_cols:
    rater_mean = np.mean(tmp_df[rater])
    rater_accs.append(rater_mean)

  acc_mean = np.mean(rater_accs)
  acc_std = np.std(rater_accs)

  label = label.lower()

  print(f"Avg receiver prediction accuracy of {label} samples across raters: mean={acc_mean}, std={acc_std}.")

print(f"Avg receiver prediction accuracy of all samples: mean={np.mean(receiver_df['mean'])}, std={np.std(receiver_df['mean'])}")

In [ ]:
#@title Fig 4 (B.1) score distributions of real and generated samples

fig = plt.figure(figsize =(5, 3))
ax = fig.add_subplot(111)

receiver_data = [receiver_df[receiver_df['label'] == 'Real']['mean'].to_numpy(), receiver_df[receiver_df['label'] == 'Generated']['mean'].to_numpy()]
zvalue, pvalue = ztest(receiver_data[0], receiver_data[1], value=0)
print(f'Significance test for the accuracies of receiver predictions between real and synthetic samples: z={zvalue}, p={pvalue}.')

g = sns.violinplot(receiver_df, x='label', y='mean', inner=None, order=['Generated', 'Real'], hue='label')

g.set_xlabel('')
g.set_ylabel('Mean score')
g.set_xticklabels(['Generated', 'Real'], fontdict={'fontsize': 10, 'fontweight': 'heavy'})
plt.ylim(-0.05, 1.25)

plt.yticks(ticks=np.arange(0., 1.2, 0.2))

ax.set_title('(B.1)', fontdict={'fontsize': 10, 'fontweight': 'heavy'})
ax.set_xlabel('Sample type', fontdict={'fontsize': 10, 'fontweight': 'heavy'})
ax.set_ylabel('Mean sample rating', fontdict={'fontsize': 10, 'fontweight': 'heavy'})


In [ ]:
#@title Fig 4 (B.2)

receiver_df_val_counts = receiver_df[['label', 'mean']].value_counts().reset_index()
receiver_df_val_counts = receiver_df_val_counts.rename({0: 'proportion'}, axis=1)

generated_num = np.where(receiver_df['label'] == 'Generated')[0].size
real_num = np.where(receiver_df['label'] == 'Real')[0].size


def calc_proportion(row):
  if row['label'] == 'Real':
    row['proportion'] /= real_num
  else:
    row['proportion'] /= generated_num

  return row

receiver_df_val_counts = receiver_df_val_counts.apply(lambda x: calc_proportion(x), axis=1)

receiver_df_val_counts

missing_data = [
    dict(label='Generated', mean=0.0, proportion=0.0),
    dict(label='Real', mean=0.0, proportion=0.0),
    dict(label='Generated', mean=0.2, proportion=0.0),
    dict(label='Real', mean=0.2, proportion=0.0)]

for md in missing_data:
  receiver_df_val_counts = pd.concat([receiver_df_val_counts, pd.DataFrame([md])], ignore_index=True)


fig = plt.figure(figsize =(5, 3))
ax = fig.add_subplot(111)
bins = [0.1, 0.3, 0.5, 0.7, 0.9, 1.1]
_df = receiver_df_val_counts[['mean', 'label', 'proportion']]

hue_order = ['Generated', 'Real']

bp = sns.barplot(_df, x='mean', y='proportion', hue='label', hue_order=hue_order)
ax.get_legend().set_title('Sample type')
sns.move_legend(bp, "upper left")

ax.xaxis.grid(False)
ax.set_ylabel('Sample proportion', fontdict={'fontsize': 10, 'fontweight': 'heavy'})
ax.set_title('(B.2)', fontdict={'fontsize': 10, 'fontweight': 'heavy'})
ax.set_xlabel('Mean rating value', fontdict={'fontsize': 10, 'fontweight': 'heavy'})


In [ ]:
#@title Fig 4.C Diversity of raters

print("Diversity of the raters")
fig = plt.figure(figsize =(5, 3))
ax = fig.add_subplot(111)

sns.violinplot(receiver_df[rater_cols], inner=None)

fvalue, pvalue = stats.f_oneway(receiver_df['A'], receiver_df['B'], receiver_df['C'], receiver_df['D'], receiver_df['E'])
print(f"Whole group: F={fvalue}, p={pvalue}")

for a, b in [['A', 'E'], ['B', 'D']]:
  zvalue, pvalue = ztest(receiver_df[a], receiver_df[b], value=0)
  print(f'{a} vs {b}: z={zvalue}, p={pvalue}')

ax.set_title('(C)', fontdict={'fontsize': 10, 'fontweight': 'heavy'})
ax.set_xlabel('Rater', fontdict={'fontsize': 10, 'fontweight': 'heavy'})
ax.set_ylabel('Sample rating', fontdict={'fontsize': 10, 'fontweight': 'heavy'})
plt.yticks(ticks=np.arange(0.0, 1.2, 1.))


In [ ]:
#@title Load raw data of Task 3

retrieval_df_str = 'UEFSMRUEFSAVJEwVBBUAEgAAEDwAAAAAAADwPwAAAAAAAAAAFQAVHhUiLBVkFRAVBhUGHBgIAAAAAAAA8D8YCAAAAAAAAACAFgAoCAAAAAAAAPA/GAgAAAAAAAAAgAAAAA84AgAAAGQBAQ/cl37O3c4BJuQBHBUKGTUABhAZGAFBFQIWZBbUARbcASZIJggcGAgAAAAAAADwPxgIAAAAAAAAAIAWACgIAAAAAAAA8D8YCAAAAAAAAACAABksFQQVABUCABUAFRAVAgAAABUEFSAVJEwVBBUAEgAAEDwAAAAAAAAAAAAAAAAAAPA/FQAVHhUiLBVkFRAVBhUGHBgIAAAAAAAA8D8YCAAAAAAAAACAFgAoCAAAAAAAAPA/GAgAAAAAAAAAgAAAAA84AgAAAGQBAQ9iSSm8InUCJvYEHBUKGTUABhAZGAFCFQIWZBbUARbcASbaAyaaAxwYCAAAAAAAAPA/GAgAAAAAAAAAgBYAKAgAAAAAAADwPxgIAAAAAAAAAIAAGSwVBBUAFQIAFQAVEBUCAAAAFQQVIBUkTBUEFQASAAAQPAAAAAAAAAAAAAAAAAAA8D8VABUeFSIsFWQVEBUGFQYcGAgAAAAAAADwPxgIAAAAAAAAAIAWACgIAAAAAAAA8D8YCAAAAAAAAACAAAAADzgCAAAAZAEBD/Z7KRAKdwEmjAgcFQoZNQAGEBkYAUMVAhZkFtQBFtwBJvAGJrAGHBgIAAAAAAAA8D8YCAAAAAAAAACAFgAoCAAAAAAAAPA/GAgAAAAAAAAAgAAZLBUEFQAVAgAVABUQFQIAAAAVBBUgFSRMFQQVABIAABA8AAAAAAAA8D8AAAAAAAAAABUAFR4VIiwVZBUQFQYVBhwYCAAAAAAAAPA/GAgAAAAAAAAAgBYAKAgAAAAAAADwPxgIAAAAAAAAAIAAAAAPOAIAAABkAQEUAAulr1A1YyaiCxwVChk1AAYQGRgBRBUCFmQW1AEW3AEmhgomxgkcGAgAAAAAAADwPxgIAAAAAAAAAIAWACgIAAAAAAAA8D8YCAAAAAAAAACAABksFQQVABUCABUAFRAVAgAAABUEFSAVJEwVBBUAEgAAEDwAAAAAAAAAAAAAAAAAAPA/FQAVHhUiLBVkFRAVBhUGHBgIAAAAAAAA8D8YCAAAAAAAAACAFgAoCAAAAAAAAPA/GAgAAAAAAAAAgAAAAA84AgAAAGQBAQ/2Sa23AmkCJrgOHBUKGTUABhAZGAFFFQIWZBbUARbcASacDSbcDBwYCAAAAAAAAPA/GAgAAAAAAAAAgBYAKAgAAAAAAADwPxgIAAAAAAAAAIAAGSwVBBUAFQIAFQAVEBUCAAAAFQQVYBVMTBUMFQASAAAwBJqZAQEI2T8ABQEI8D8zBQEE4z8JGADJDQgk6T8AAAAAAAAAABUAFToVPiwVZBUQFQYVBhwYCAAAAAAAAPA/GAgAAAAAAAAAgBYAKAgAAAAAAADwPxgIAAAAAAAAAIAAAAAdcAIAAABkAQMPiKZQRDOl6VQNmhBWTVG1AcawIwAAJpISHBUKGTUABhAZGARtZWFuFQIWZBawAhagAibaECbyDxwYCAAAAAAAAPA/GAgAAAAAAAAAgBYAKAgAAAAAAADwPxgIAAAAAAAAAIAAGSwVBBUAFQIAFQAVEBUCAAAAFQQVMBU0TBUEFQASAAAYXAgAAABCYXNlbGluZQgAAABUYWN0aWNBSRUAFSIVJiwVZBUQFQYVBhw2ACgIVGFjdGljQUkYCEJhc2VsaW5lAAAAEUACAAAAZAEBA0IQAQuYuErzAiaaFRwVDBk1AAYQGRgFbGFiZWwVAhZkFsABFsgBJqIUJtITHDYAKAhUYWN0aWNBSRgIQmFzZWxpbmUAGSwVBBUAFQIAFQAVEBUCAAAAFQQZjDUAGAZzY2hlbWEVDgAVCiUCGAFBABUKJQIYAUIAFQolAhgBQwAVCiUCGAFEABUKJQIYAUUAFQolAhgEbWVhbgAVDCUCGAVsYWJlbCUATBwAAAAWZBkcGXwm5AEcFQoZNQAGEBkYAUEVAhZkFtQBFtwBJkgmCBwYCAAAAAAAAPA/GAgAAAAAAAAAgBYAKAgAAAAAAADwPxgIAAAAAAAAAIAAGSwVBBUAFQIAFQAVEBUCAAAAJvYEHBUKGTUABhAZGAFCFQIWZBbUARbcASbaAyaaAxwYCAAAAAAAAPA/GAgAAAAAAAAAgBYAKAgAAAAAAADwPxgIAAAAAAAAAIAAGSwVBBUAFQIAFQAVEBUCAAAAJowIHBUKGTUABhAZGAFDFQIWZBbUARbcASbwBiawBhwYCAAAAAAAAPA/GAgAAAAAAAAAgBYAKAgAAAAAAADwPxgIAAAAAAAAAIAAGSwVBBUAFQIAFQAVEBUCAAAAJqILHBUKGTUABhAZGAFEFQIWZBbUARbcASaGCibGCRwYCAAAAAAAAPA/GAgAAAAAAAAAgBYAKAgAAAAAAADwPxgIAAAAAAAAAIAAGSwVBBUAFQIAFQAVEBUCAAAAJrgOHBUKGTUABhAZGAFFFQIWZBbUARbcASacDSbcDBwYCAAAAAAAAPA/GAgAAAAAAAAAgBYAKAgAAAAAAADwPxgIAAAAAAAAAIAAGSwVBBUAFQIAFQAVEBUCAAAAJpISHBUKGTUABhAZGARtZWFuFQIWZBawAhagAibaECbyDxwYCAAAAAAAAPA/GAgAAAAAAAAAgBYAKAgAAAAAAADwPxgIAAAAAAAAAIAAGSwVBBUAFQIAFQAVEBUCAAAAJpoVHBUMGTUABhAZGAVsYWJlbBUCFmQWwAEWyAEmohQm0hMcNgAoCFRhY3RpY0FJGAhCYXNlbGluZQAZLBUEFQAVAgAVABUQFQIAAAAWlAwWZCYIFrQMFAAAGSwYBnBhbmRhcxiiCHsiaW5kZXhfY29sdW1ucyI6IFt7ImtpbmQiOiAicmFuZ2UiLCAibmFtZSI6IG51bGwsICJzdGFydCI6IDAsICJzdG9wIjogNTAsICJzdGVwIjogMX1dLCAiY29sdW1uX2luZGV4ZXMiOiBbeyJuYW1lIjogbnVsbCwgImZpZWxkX25hbWUiOiBudWxsLCAicGFuZGFzX3R5cGUiOiAidW5pY29kZSIsICJudW1weV90eXBlIjogIm9iamVjdCIsICJtZXRhZGF0YSI6IHsiZW5jb2RpbmciOiAiVVRGLTgifX1dLCAiY29sdW1ucyI6IFt7Im5hbWUiOiAiQSIsICJmaWVsZF9uYW1lIjogIkEiLCAicGFuZGFzX3R5cGUiOiAiZmxvYXQ2NCIsICJudW1weV90eXBlIjogImZsb2F0NjQiLCAibWV0YWRhdGEiOiBudWxsfSwgeyJuYW1lIjogIkIiLCAiZmllbGRfbmFtZSI6ICJCIiwgInBhbmRhc190eXBlIjogImZsb2F0NjQiLCAibnVtcHlfdHlwZSI6ICJmbG9hdDY0IiwgIm1ldGFkYXRhIjogbnVsbH0sIHsibmFtZSI6ICJDIiwgImZpZWxkX25hbWUiOiAiQyIsICJwYW5kYXNfdHlwZSI6ICJmbG9hdDY0IiwgIm51bXB5X3R5cGUiOiAiZmxvYXQ2NCIsICJtZXRhZGF0YSI6IG51bGx9LCB7Im5hbWUiOiAiRCIsICJmaWVsZF9uYW1lIjogIkQiLCAicGFuZGFzX3R5cGUiOiAiZmxvYXQ2NCIsICJudW1weV90eXBlIjogImZsb2F0NjQiLCAibWV0YWRhdGEiOiBudWxsfSwgeyJuYW1lIjogIkUiLCAiZmllbGRfbmFtZSI6ICJFIiwgInBhbmRhc190eXBlIjogImZsb2F0NjQiLCAibnVtcHlfdHlwZSI6ICJmbG9hdDY0IiwgIm1ldGFkYXRhIjogbnVsbH0sIHsibmFtZSI6ICJtZWFuIiwgImZpZWxkX25hbWUiOiAibWVhbiIsICJwYW5kYXNfdHlwZSI6ICJmbG9hdDY0IiwgIm51bXB5X3R5cGUiOiAiZmxvYXQ2NCIsICJtZXRhZGF0YSI6IG51bGx9LCB7Im5hbWUiOiAibGFiZWwiLCAiZmllbGRfbmFtZSI6ICJsYWJlbCIsICJwYW5kYXNfdHlwZSI6ICJ1bmljb2RlIiwgIm51bXB5X3R5cGUiOiAib2JqZWN0IiwgIm1ldGFkYXRhIjogbnVsbH1dLCAiY3JlYXRvciI6IHsibGlicmFyeSI6ICJweWFycm93IiwgInZlcnNpb24iOiAiMTMuMC4wIn0sICJwYW5kYXNfdmVyc2lvbiI6ICIyLjAuMyJ9ABgMQVJST1c6c2NoZW1hGNgPLy8vLy85Z0ZBQUFRQUFBQUFBQUtBQTRBQmdBRkFBZ0FDZ0FBQUFBQkJBQVFBQUFBQUFBS0FBd0FBQUFFQUFnQUNnQUFBRmdFQUFBRUFBQUFBUUFBQUF3QUFBQUlBQXdBQkFBSUFBZ0FBQUF3QkFBQUJBQUFBQ0lFQUFCN0ltbHVaR1Y0WDJOdmJIVnRibk1pT2lCYmV5SnJhVzVrSWpvZ0luSmhibWRsSWl3Z0ltNWhiV1VpT2lCdWRXeHNMQ0FpYzNSaGNuUWlPaUF3TENBaWMzUnZjQ0k2SURVd0xDQWljM1JsY0NJNklERjlYU3dnSW1OdmJIVnRibDlwYm1SbGVHVnpJam9nVzNzaWJtRnRaU0k2SUc1MWJHd3NJQ0ptYVdWc1pGOXVZVzFsSWpvZ2JuVnNiQ3dnSW5CaGJtUmhjMTkwZVhCbElqb2dJblZ1YVdOdlpHVWlMQ0FpYm5WdGNIbGZkSGx3WlNJNklDSnZZbXBsWTNRaUxDQWliV1YwWVdSaGRHRWlPaUI3SW1WdVkyOWthVzVuSWpvZ0lsVlVSaTA0SW4xOVhTd2dJbU52YkhWdGJuTWlPaUJiZXlKdVlXMWxJam9nSWtFaUxDQWlabWxsYkdSZmJtRnRaU0k2SUNKQklpd2dJbkJoYm1SaGMxOTBlWEJsSWpvZ0ltWnNiMkYwTmpRaUxDQWliblZ0Y0hsZmRIbHdaU0k2SUNKbWJHOWhkRFkwSWl3Z0ltMWxkR0ZrWVhSaElqb2diblZzYkgwc0lIc2libUZ0WlNJNklDSkNJaXdnSW1acFpXeGtYMjVoYldVaU9pQWlRaUlzSUNKd1lXNWtZWE5mZEhsd1pTSTZJQ0ptYkc5aGREWTBJaXdnSW01MWJYQjVYM1I1Y0dVaU9pQWlabXh2WVhRMk5DSXNJQ0p0WlhSaFpHRjBZU0k2SUc1MWJHeDlMQ0I3SW01aGJXVWlPaUFpUXlJc0lDSm1hV1ZzWkY5dVlXMWxJam9nSWtNaUxDQWljR0Z1WkdGelgzUjVjR1VpT2lBaVpteHZZWFEyTkNJc0lDSnVkVzF3ZVY5MGVYQmxJam9nSW1ac2IyRjBOalFpTENBaWJXVjBZV1JoZEdFaU9pQnVkV3hzZlN3Z2V5SnVZVzFsSWpvZ0lrUWlMQ0FpWm1sbGJHUmZibUZ0WlNJNklDSkVJaXdnSW5CaGJtUmhjMTkwZVhCbElqb2dJbVpzYjJGME5qUWlMQ0FpYm5WdGNIbGZkSGx3WlNJNklDSm1iRzloZERZMElpd2dJbTFsZEdGa1lYUmhJam9nYm5Wc2JIMHNJSHNpYm1GdFpTSTZJQ0pGSWl3Z0ltWnBaV3hrWDI1aGJXVWlPaUFpUlNJc0lDSndZVzVrWVhOZmRIbHdaU0k2SUNKbWJHOWhkRFkwSWl3Z0ltNTFiWEI1WDNSNWNHVWlPaUFpWm14dllYUTJOQ0lzSUNKdFpYUmhaR0YwWVNJNklHNTFiR3g5TENCN0ltNWhiV1VpT2lBaWJXVmhiaUlzSUNKbWFXVnNaRjl1WVcxbElqb2dJbTFsWVc0aUxDQWljR0Z1WkdGelgzUjVjR1VpT2lBaVpteHZZWFEyTkNJc0lDSnVkVzF3ZVY5MGVYQmxJam9nSW1ac2IyRjBOalFpTENBaWJXVjBZV1JoZEdFaU9pQnVkV3hzZlN3Z2V5SnVZVzFsSWpvZ0lteGhZbVZzSWl3Z0ltWnBaV3hrWDI1aGJXVWlPaUFpYkdGaVpXd2lMQ0FpY0dGdVpHRnpYM1I1Y0dVaU9pQWlkVzVwWTI5a1pTSXNJQ0p1ZFcxd2VWOTBlWEJsSWpvZ0ltOWlhbVZqZENJc0lDSnRaWFJoWkdGMFlTSTZJRzUxYkd4OVhTd2dJbU55WldGMGIzSWlPaUI3SW14cFluSmhjbmtpT2lBaWNIbGhjbkp2ZHlJc0lDSjJaWEp6YVc5dUlqb2dJakV6TGpBdU1DSjlMQ0FpY0dGdVpHRnpYM1psY25OcGIyNGlPaUFpTWk0d0xqTWlmUUFBQmdBQUFIQmhibVJoY3dBQUJ3QUFBQ1FCQUFEb0FBQUF2QUFBQUpBQUFBQmtBQUFBTkFBQUFBUUFBQUFJLy8vL0FBQUJCUkFBQUFBY0FBQUFCQUFBQUFBQUFBQUZBQUFBYkdGaVpXd0FBQUFFQUFRQUJBQUFBRFQvLy84QUFBRURFQUFBQUJnQUFBQUVBQUFBQUFBQUFBUUFBQUJ0WldGdUFBQUFBQ3IvLy84QUFBSUFZUC8vL3dBQUFRTVFBQUFBRkFBQUFBUUFBQUFBQUFBQUFRQUFBRVVBQUFCUy8vLy9BQUFDQUlqLy8vOEFBQUVERUFBQUFCUUFBQUFFQUFBQUFBQUFBQUVBQUFCRUFBQUFldi8vL3dBQUFnQ3cvLy8vQUFBQkF4QUFBQUFVQUFBQUJBQUFBQUFBQUFBQkFBQUFRd0FBQUtMLy8vOEFBQUlBMlAvLy93QUFBUU1RQUFBQUZBQUFBQVFBQUFBQUFBQUFBUUFBQUVJQUFBREsvLy8vQUFBQ0FCQUFGQUFJQUFZQUJ3QU1BQUFBRUFBUUFBQUFBQUFCQXhBQUFBQVlBQUFBQkFBQUFBQUFBQUFCQUFBQVFRQUdBQWdBQmdBR0FBQUFBQUFDQUE9PQAYIHBhcnF1ZXQtY3BwLWFycm93IHZlcnNpb24gMTMuMC4wGXwcAAAcAAAcAAAcAAAcAAAcAAAcAAAAOQ8AAFBBUjE='
retrieval_df = pd.read_parquet(io.BytesIO(base64.b64decode(retrieval_df_str)))
del retrieval_df_str
retrieval_df.info()
retrieval_df.head()

In [ ]:
#@title Top-1 accuracies for real and synthetic samples

for label in ['Baseline', 'TacticAI']:
  tmp_df = retrieval_df[retrieval_df['label'] == label]
  rater_accs = []

  for rater in rater_cols:
    rater_mean = np.mean(tmp_df[rater])
    rater_accs.append(rater_mean)

  acc_mean = np.mean(rater_accs)
  acc_std = np.std(rater_accs)

  label = label.lower()

  print(f"Top-1 retrieval accuracy of {label} samples across raters: mean={acc_mean}, std={acc_std}.")

retrieval_data = [retrieval_df[retrieval_df['label'] == 'TacticAI']['mean'].to_numpy(), retrieval_df[retrieval_df['label'] == 'Baseline']['mean'].to_numpy()]
zvalue, pvalue = ztest(retrieval_data[0], retrieval_data[1], value=0)
print(f'Statistical test for the difference between TacticAI and Baseline: z={zvalue}, p={pvalue}.')

In [ ]:
#@title Fig 4.D Inter-agreement for TacticAI retrievals
plt.figure(figsize=(5, 3))
retrieval_df2 = retrieval_df[retrieval_df['label'] == 'TacticAI']
fvalue, pvalue = stats.f_oneway(retrieval_df2['A'], retrieval_df2['B'], retrieval_df2['C'], retrieval_df2['D'], retrieval_df2['E'])
print(f'Inter-agreement of TacticAI retrievals: F={fvalue}, p={pvalue}.')

sns.violinplot(retrieval_df2[rater_cols], inner=None)

plt.yticks(ticks=np.arange(0., 1.2, 1.))

plt.gca().set_xlabel('Rater', fontdict={'fontsize': 10, 'fontweight': 'heavy'})
plt.gca().set_ylabel('Sample rating', fontdict={'fontsize': 10, 'fontweight': 'heavy'})
plt.gca().set_title('(D)', fontdict={'fontsize': 10, 'fontweight': 'heavy'})


In [ ]:
#@title Load raw data for Task 4.

recsys_df_str = 'UEFSMRUEFUAVNkwVCBUAEgAAIAAABQEE8D8FBwgAAIAJCCTwvwAAAAAAAAAAFQAVLhUyLBVkFRAVBhUGHBgIAAAAAAAA8D8YCAAAAAAAAPC/FgAoCAAAAAAAAPA/GAgAAAAAAADwvwAAABdYAgAAAGQBAgMEABAAC4LgQMAjgS8AAQAmhgIcFQoZNQAGEBkYAUEVAhZkFoQCFv4BJlomCBwYCAAAAAAAAPA/GAgAAAAAAADwvxYAKAgAAAAAAADwPxgIAAAAAAAA8L8AGSwVBBUAFQIAFQAVEBUCAAAAFQQVMBUsTBUGFQASAAAYAAAFAQTwPwUHKADwvwAAAAAAAAAAFQAVKBUsLBVkFRAVBhUGHBgIAAAAAAAA8D8YCAAAAAAAAPC/FgAoCAAAAAAAAPA/GAgAAAAAAADwvwAAABRMAgAAAGQBAiAAC0EAAIAAQQEAAQAmqgUcFQoZNQAGEBkYAUIVAhZkFu4BFu4BJoQEJrwDHBgIAAAAAAAA8D8YCAAAAAAAAPC/FgAoCAAAAAAAAPA/GAgAAAAAAADwvwAZLBUEFQAVAgAVABUQFQIAAAAVBBUwFSxMFQYVABIAABgAAAUBBPA/BQcoAPC/AAAAAAAAAIAVABUoFSwsFWQVEBUGFQYcGAgAAAAAAADwPxgIAAAAAAAA8L8WACgIAAAAAAAA8D8YCAAAAAAAAPC/AAAAFEwCAAAAZAECIAALAUAABABBBAgBACbSCBwVChk1AAYQGRgBQxUCFmQW7gEW7gEmrAcm5AYcGAgAAAAAAADwPxgIAAAAAAAA8L8WACgIAAAAAAAA8D8YCAAAAAAAAPC/ABksFQQVABUCABUAFRAVAgAAABUEFUAVNkwVCBUAEgAAIAAABQEE8D8FBwgA8L8JCCQAAAAAAAAAAACAFQAVLBUwLBVkFRAVBhUGHBgIAAAAAAAA8D8YCAAAAAAAAPC/FgAoCAAAAAAAAPA/GAgAAAAAAADwvwAAABZUAgAAAGQBAg8EAAQAQYAAAwQDOAQBACaIDBwVChk1AAYQGRgBRBUCFmQWggIW/AEm3gomjAocGAgAAAAAAADwPxgIAAAAAAAA8L8WACgIAAAAAAAA8D8YCAAAAAAAAPC/ABksFQQVABUCABUAFRAVAgAAABUEFUAVNkwVCBUAEgAAIAAABQEE8D8FBwgAAIAJCCTwvwAAAAAAAAAAFQAVKBUsLBVkFRAVBhUGHBgIAAAAAAAA8D8YCAAAAAAAAPC/FgAoCAAAAAAAAPA/GAgAAAAAAADwvwAAABRMAgAAAGQBAiAAC4EgAIAgjgMAAAAmug8cFQoZNQAGEBkYAUUVAhZkFv4BFvgBJpQOJsINHBgIAAAAAAAA8D8YCAAAAAAAAPC/FgAoCAAAAAAAAPA/GAgAAAAAAADwvwAZLBUEFQAVAgAVABUQFQIAAAAVBBUqFS5MFQQVABIAABVQBAAAAFJlYWwJAAAAR2VuZXJhdGVkFQAVHhUiLBVkFRAVBhUGHDYAKARSZWFsGAlHZW5lcmF0ZWQAAAAPOAIAAABkAQEPFmIpedLkASasEhwVDBk1AAYQGRgFbGFiZWwVAhZkFrABFrgBJr4RJvQQHDYAKARSZWFsGAlHZW5lcmF0ZWQAGSwVBBUAFQIAFQAVEBUCAAAAFQQVgAEVXkwVEBUAEgAAQAAABQEM8D+amQEBCNk/MwUBAOMNEATpvwUPCDPjvwkgAMkRGCA/mpmZmZmZ2b8VABU6FT4sFWQVEBUGFQYcGAgAAAAAAADwPxgImpmZmZmZ6b8WACgIAAAAAAAA8D8YCJqZmZmZmem/AAAAHXACAAAAZAEDDwgAABAAAAMItABsoVZBg22AAAcAACb0FRwVChk1AAYQGRgEbWVhbhUCFmQW0gIWtAImvBQmwBMcGAgAAAAAAADwPxgImpmZmZmZ6b8WACgIAAAAAAAA8D8YCJqZmZmZmem/ABksFQQVABUCABUAFRAVAgAAABUEGYw1ABgGc2NoZW1hFQ4AFQolAhgBQQAVCiUCGAFCABUKJQIYAUMAFQolAhgBRAAVCiUCGAFFABUMJQIYBWxhYmVsJQBMHAAAABUKJQIYBG1lYW4AFmQZHBl8JoYCHBUKGTUABhAZGAFBFQIWZBaEAhb+ASZaJggcGAgAAAAAAADwPxgIAAAAAAAA8L8WACgIAAAAAAAA8D8YCAAAAAAAAPC/ABksFQQVABUCABUAFRAVAgAAACaqBRwVChk1AAYQGRgBQhUCFmQW7gEW7gEmhAQmvAMcGAgAAAAAAADwPxgIAAAAAAAA8L8WACgIAAAAAAAA8D8YCAAAAAAAAPC/ABksFQQVABUCABUAFRAVAgAAACbSCBwVChk1AAYQGRgBQxUCFmQW7gEW7gEmrAcm5AYcGAgAAAAAAADwPxgIAAAAAAAA8L8WACgIAAAAAAAA8D8YCAAAAAAAAPC/ABksFQQVABUCABUAFRAVAgAAACaIDBwVChk1AAYQGRgBRBUCFmQWggIW/AEm3gomjAocGAgAAAAAAADwPxgIAAAAAAAA8L8WACgIAAAAAAAA8D8YCAAAAAAAAPC/ABksFQQVABUCABUAFRAVAgAAACa6DxwVChk1AAYQGRgBRRUCFmQW/gEW+AEmlA4mwg0cGAgAAAAAAADwPxgIAAAAAAAA8L8WACgIAAAAAAAA8D8YCAAAAAAAAPC/ABksFQQVABUCABUAFRAVAgAAACasEhwVDBk1AAYQGRgFbGFiZWwVAhZkFrABFrgBJr4RJvQQHDYAKARSZWFsGAlHZW5lcmF0ZWQAGSwVBBUAFQIAFQAVEBUCAAAAJvQVHBUKGTUABhAZGARtZWFuFQIWZBbSAha0Aia8FCbAExwYCAAAAAAAAPA/GAiamZmZmZnpvxYAKAgAAAAAAADwPxgImpmZmZmZ6b8AGSwVBBUAFQIAFQAVEBUCAAAAFuINFmQmCBa6DRQAABksGAZwYW5kYXMYogh7ImluZGV4X2NvbHVtbnMiOiBbeyJraW5kIjogInJhbmdlIiwgIm5hbWUiOiBudWxsLCAic3RhcnQiOiAwLCAic3RvcCI6IDUwLCAic3RlcCI6IDF9XSwgImNvbHVtbl9pbmRleGVzIjogW3sibmFtZSI6IG51bGwsICJmaWVsZF9uYW1lIjogbnVsbCwgInBhbmRhc190eXBlIjogInVuaWNvZGUiLCAibnVtcHlfdHlwZSI6ICJvYmplY3QiLCAibWV0YWRhdGEiOiB7ImVuY29kaW5nIjogIlVURi04In19XSwgImNvbHVtbnMiOiBbeyJuYW1lIjogIkEiLCAiZmllbGRfbmFtZSI6ICJBIiwgInBhbmRhc190eXBlIjogImZsb2F0NjQiLCAibnVtcHlfdHlwZSI6ICJmbG9hdDY0IiwgIm1ldGFkYXRhIjogbnVsbH0sIHsibmFtZSI6ICJCIiwgImZpZWxkX25hbWUiOiAiQiIsICJwYW5kYXNfdHlwZSI6ICJmbG9hdDY0IiwgIm51bXB5X3R5cGUiOiAiZmxvYXQ2NCIsICJtZXRhZGF0YSI6IG51bGx9LCB7Im5hbWUiOiAiQyIsICJmaWVsZF9uYW1lIjogIkMiLCAicGFuZGFzX3R5cGUiOiAiZmxvYXQ2NCIsICJudW1weV90eXBlIjogImZsb2F0NjQiLCAibWV0YWRhdGEiOiBudWxsfSwgeyJuYW1lIjogIkQiLCAiZmllbGRfbmFtZSI6ICJEIiwgInBhbmRhc190eXBlIjogImZsb2F0NjQiLCAibnVtcHlfdHlwZSI6ICJmbG9hdDY0IiwgIm1ldGFkYXRhIjogbnVsbH0sIHsibmFtZSI6ICJFIiwgImZpZWxkX25hbWUiOiAiRSIsICJwYW5kYXNfdHlwZSI6ICJmbG9hdDY0IiwgIm51bXB5X3R5cGUiOiAiZmxvYXQ2NCIsICJtZXRhZGF0YSI6IG51bGx9LCB7Im5hbWUiOiAibGFiZWwiLCAiZmllbGRfbmFtZSI6ICJsYWJlbCIsICJwYW5kYXNfdHlwZSI6ICJ1bmljb2RlIiwgIm51bXB5X3R5cGUiOiAib2JqZWN0IiwgIm1ldGFkYXRhIjogbnVsbH0sIHsibmFtZSI6ICJtZWFuIiwgImZpZWxkX25hbWUiOiAibWVhbiIsICJwYW5kYXNfdHlwZSI6ICJmbG9hdDY0IiwgIm51bXB5X3R5cGUiOiAiZmxvYXQ2NCIsICJtZXRhZGF0YSI6IG51bGx9XSwgImNyZWF0b3IiOiB7ImxpYnJhcnkiOiAicHlhcnJvdyIsICJ2ZXJzaW9uIjogIjEzLjAuMCJ9LCAicGFuZGFzX3ZlcnNpb24iOiAiMi4wLjMifQAYDEFSUk9XOnNjaGVtYRjYDy8vLy8vOWdGQUFBUUFBQUFBQUFLQUE0QUJnQUZBQWdBQ2dBQUFBQUJCQUFRQUFBQUFBQUtBQXdBQUFBRUFBZ0FDZ0FBQUZnRUFBQUVBQUFBQVFBQUFBd0FBQUFJQUF3QUJBQUlBQWdBQUFBd0JBQUFCQUFBQUNJRUFBQjdJbWx1WkdWNFgyTnZiSFZ0Ym5NaU9pQmJleUpyYVc1a0lqb2dJbkpoYm1kbElpd2dJbTVoYldVaU9pQnVkV3hzTENBaWMzUmhjblFpT2lBd0xDQWljM1J2Y0NJNklEVXdMQ0FpYzNSbGNDSTZJREY5WFN3Z0ltTnZiSFZ0Ymw5cGJtUmxlR1Z6SWpvZ1czc2libUZ0WlNJNklHNTFiR3dzSUNKbWFXVnNaRjl1WVcxbElqb2diblZzYkN3Z0luQmhibVJoYzE5MGVYQmxJam9nSW5WdWFXTnZaR1VpTENBaWJuVnRjSGxmZEhsd1pTSTZJQ0p2WW1wbFkzUWlMQ0FpYldWMFlXUmhkR0VpT2lCN0ltVnVZMjlrYVc1bklqb2dJbFZVUmkwNEluMTlYU3dnSW1OdmJIVnRibk1pT2lCYmV5SnVZVzFsSWpvZ0lrRWlMQ0FpWm1sbGJHUmZibUZ0WlNJNklDSkJJaXdnSW5CaGJtUmhjMTkwZVhCbElqb2dJbVpzYjJGME5qUWlMQ0FpYm5WdGNIbGZkSGx3WlNJNklDSm1iRzloZERZMElpd2dJbTFsZEdGa1lYUmhJam9nYm5Wc2JIMHNJSHNpYm1GdFpTSTZJQ0pDSWl3Z0ltWnBaV3hrWDI1aGJXVWlPaUFpUWlJc0lDSndZVzVrWVhOZmRIbHdaU0k2SUNKbWJHOWhkRFkwSWl3Z0ltNTFiWEI1WDNSNWNHVWlPaUFpWm14dllYUTJOQ0lzSUNKdFpYUmhaR0YwWVNJNklHNTFiR3g5TENCN0ltNWhiV1VpT2lBaVF5SXNJQ0ptYVdWc1pGOXVZVzFsSWpvZ0lrTWlMQ0FpY0dGdVpHRnpYM1I1Y0dVaU9pQWlabXh2WVhRMk5DSXNJQ0p1ZFcxd2VWOTBlWEJsSWpvZ0ltWnNiMkYwTmpRaUxDQWliV1YwWVdSaGRHRWlPaUJ1ZFd4c2ZTd2dleUp1WVcxbElqb2dJa1FpTENBaVptbGxiR1JmYm1GdFpTSTZJQ0pFSWl3Z0luQmhibVJoYzE5MGVYQmxJam9nSW1ac2IyRjBOalFpTENBaWJuVnRjSGxmZEhsd1pTSTZJQ0ptYkc5aGREWTBJaXdnSW0xbGRHRmtZWFJoSWpvZ2JuVnNiSDBzSUhzaWJtRnRaU0k2SUNKRklpd2dJbVpwWld4a1gyNWhiV1VpT2lBaVJTSXNJQ0p3WVc1a1lYTmZkSGx3WlNJNklDSm1iRzloZERZMElpd2dJbTUxYlhCNVgzUjVjR1VpT2lBaVpteHZZWFEyTkNJc0lDSnRaWFJoWkdGMFlTSTZJRzUxYkd4OUxDQjdJbTVoYldVaU9pQWliR0ZpWld3aUxDQWlabWxsYkdSZmJtRnRaU0k2SUNKc1lXSmxiQ0lzSUNKd1lXNWtZWE5mZEhsd1pTSTZJQ0oxYm1samIyUmxJaXdnSW01MWJYQjVYM1I1Y0dVaU9pQWliMkpxWldOMElpd2dJbTFsZEdGa1lYUmhJam9nYm5Wc2JIMHNJSHNpYm1GdFpTSTZJQ0p0WldGdUlpd2dJbVpwWld4a1gyNWhiV1VpT2lBaWJXVmhiaUlzSUNKd1lXNWtZWE5mZEhsd1pTSTZJQ0ptYkc5aGREWTBJaXdnSW01MWJYQjVYM1I1Y0dVaU9pQWlabXh2WVhRMk5DSXNJQ0p0WlhSaFpHRjBZU0k2SUc1MWJHeDlYU3dnSW1OeVpXRjBiM0lpT2lCN0lteHBZbkpoY25raU9pQWljSGxoY25KdmR5SXNJQ0oyWlhKemFXOXVJam9nSWpFekxqQXVNQ0o5TENBaWNHRnVaR0Z6WDNabGNuTnBiMjRpT2lBaU1pNHdMak1pZlFBQUJnQUFBSEJoYm1SaGN3QUFCd0FBQUNRQkFBRG9BQUFBdkFBQUFKQUFBQUJrQUFBQU5BQUFBQVFBQUFBSS8vLy9BQUFCQXhBQUFBQVlBQUFBQkFBQUFBQUFBQUFFQUFBQWJXVmhiZ0FBQUFEKy92Ly9BQUFDQURULy8vOEFBQUVGRUFBQUFCd0FBQUFFQUFBQUFBQUFBQVVBQUFCc1lXSmxiQUFBQUFRQUJBQUVBQUFBWVAvLy93QUFBUU1RQUFBQUZBQUFBQVFBQUFBQUFBQUFBUUFBQUVVQUFBQlMvLy8vQUFBQ0FJai8vLzhBQUFFREVBQUFBQlFBQUFBRUFBQUFBQUFBQUFFQUFBQkVBQUFBZXYvLy93QUFBZ0N3Ly8vL0FBQUJBeEFBQUFBVUFBQUFCQUFBQUFBQUFBQUJBQUFBUXdBQUFLTC8vLzhBQUFJQTJQLy8vd0FBQVFNUUFBQUFGQUFBQUFRQUFBQUFBQUFBQVFBQUFFSUFBQURLLy8vL0FBQUNBQkFBRkFBSUFBWUFCd0FNQUFBQUVBQVFBQUFBQUFBQkF4QUFBQUFZQUFBQUJBQUFBQUFBQUFBQkFBQUFRUUFHQUFnQUJnQUdBQUFBQUFBQ0FBPT0AGCBwYXJxdWV0LWNwcC1hcnJvdyB2ZXJzaW9uIDEzLjAuMBl8HAAAHAAAHAAAHAAAHAAAHAAAHAAAADYPAABQQVIx'
recsys_df = pd.read_parquet(io.BytesIO(base64.b64decode(recsys_df_str)))
del recsys_df_str
recsys_df.info()
recsys_df.head()

In [ ]:
#@title Fig 4(E) Inter-agreement of TacticAI recommendations.

favourable_samples = np.sum(recsys_df['mean'] > 0.)
print(f'Ratio of favourable recommendations by majoirty vote: {favourable_samples / len(recsys_df)}.')

fvalue, pvalue = stats.f_oneway(recsys_df['A'], recsys_df['B'], recsys_df['C'], recsys_df['D'], recsys_df['E'])
print(f'Inter-agreement of TacticAI recommendations: F={fvalue}, p={pvalue}')

plt.figure(figsize=(5, 3))
g = sns.violinplot(recsys_df[rater_cols], inner=None)
g.set_xlabel('')
plt.yticks(ticks=np.arange(-1.0, 1.2, 1.))

plt.gca().set_xlabel('Rater', fontdict={'fontsize': 10, 'fontweight': 'heavy'})
plt.gca().set_ylabel('Sample rating', fontdict={'fontsize': 10, 'fontweight': 'heavy'})
plt.gca().set_title('(E)', fontdict={'fontsize': 10, 'fontweight': 'heavy'})


In [ ]:
#@title Significance test of ratings (Null hypothesis: mean_rating = 0)

for rater in rater_cols:
  tvalue, pvalue = stats.ttest_1samp(recsys_df[rater], popmean=0)
  print(f"Significance test for the ratings by Rater {rater}: t={tvalue}, p={pvalue}.")

tvalue_avg, pvalue_avg = stats.ttest_1samp(recsys_df['mean'], popmean=0)
print(f"Significance test for the avg ratings: t={tvalue_avg}, p={pvalue_avg}.")